In [13]:
# конвертация картинок в base64
import base64
import mimetypes
from pathlib import Path

def image_to_data_url(path: Path) -> str:
    mime, _ = mimetypes.guess_type(path)
    if mime is None:
        mime = "image/jpeg"
    b64 = base64.b64encode(path.read_bytes()).decode("utf-8")
    return f"data:{mime};base64,{b64}"


In [31]:
# Подготовка датасета
from pathlib import Path
import pandas as pd
import random
import re

ROOT = Path("")
IMG_DIR = ROOT / "flickr30k_images"

SEED = 42
N_BASE = 300

OUT_BASE = ROOT / "df_diff_details.csv"
OUT_FINAL = ROOT / "df_all_images.csv"

random.seed(SEED)

COLORS = [
    "red", "blue", "green", "black", "white", "yellow",
    "orange", "pink", "purple", "brown", "violet", "gray", "grey"
]

color_pat = re.compile(r"\b(" + "|".join(COLORS) + r")\b", flags=re.IGNORECASE)
gender_pat = re.compile(r"\b(man|men|woman|women)\b", flags=re.IGNORECASE)

df = pd.read_csv(ROOT / "results.csv", sep="|")
df.columns = df.columns.str.strip()

df["comment_number"] = pd.to_numeric(df["comment_number"], errors="coerce")
df0 = df[df["comment_number"] == 0].copy()

df0["comment"] = df0["comment"].astype(str)

df0["image_path"] = df0["image_name"].apply(lambda x: str(IMG_DIR / x))
df0 = df0[df0["image_path"].apply(lambda p: Path(p).exists())].reset_index(drop=True)


def count_colors(text: str) -> int:
    return len(color_pat.findall(text))


def count_gender(text: str) -> int:
    return len(gender_pat.findall(text))


df0["color_count"] = df0["comment"].apply(count_colors)
df0["gender_count"] = df0["comment"].apply(count_gender)

color_candidates = df0[df0["color_count"] == 1].copy()
color_candidates["target_edit_type"] = "color"

gender_candidates = df0[df0["gender_count"] == 1].copy()
gender_candidates["target_edit_type"] = "gender"

candidates = pd.concat([color_candidates, gender_candidates], ignore_index=True)


def make_false_caption(true_caption: str, target_edit_type: str):
    if target_edit_type == "color":
        matches = color_pat.findall(true_caption)
        if len(matches) != 1:
            return "", "none", "", ""

        old = matches[0].lower()
        new = random.choice([c for c in COLORS if c != old])

        false_caption = re.sub(
            rf"\b{re.escape(matches[0])}\b",
            new,
            true_caption,
            count=1,
            flags=re.IGNORECASE
        )
        return false_caption, "color", old, new

    if target_edit_type == "gender":
        matches = gender_pat.findall(true_caption)
        if len(matches) != 1:
            return "", "none", "", ""

        old = matches[0].lower()
        swap = {
            "man": "woman",
            "woman": "man",
            "men": "women",
            "women": "men"
        }
        new = swap[old]

        false_caption = re.sub(
            rf"\b{re.escape(matches[0])}\b",
            new,
            true_caption,
            count=1,
            flags=re.IGNORECASE
        )
        return false_caption, "gender", old, new

    return "", "none", "", ""
 

base = candidates.sample(n=N_BASE, random_state=SEED).reset_index(drop=True)

rows = []
for _, row in base.iterrows():
    img_name = row["image_name"]
    img_path = row["image_path"]
    true_caption = row["comment"]
    target_edit_type = row["target_edit_type"]

    false_caption, edit_type, old_val, new_val = make_false_caption(true_caption, target_edit_type)

    if not false_caption:
        continue

    rows.append({
        "image_name": img_name,
        "image_path": img_path,
        "true_caption": true_caption,
        "false_caption": false_caption,
        "edit_type": edit_type,
        "old_value": old_val,
        "new_value": new_val
    })

out_base = pd.DataFrame(rows)
out_base.to_csv(OUT_BASE, index=False)


n = len(out_base)
idx = list(range(n))
perm = idx.copy()
random.shuffle(perm)

for i in range(n):
    if perm[i] == i:
        j = (i + 1) % n
        perm[i], perm[j] = perm[j], perm[i]

final_rows = []
for i, row in out_base.iterrows():
    img_name = row["image_name"]
    img_path = row["image_path"]
    true_desc = row["true_caption"]

    false_detail = row["false_caption"]
    false_mismatch = out_base.loc[perm[i], "true_caption"]
    false_src_name = out_base.loc[perm[i], "image_name"]

    final_rows.append({
        "tag": "concurrence",
        "image_name": img_name,
        "image_data_url": image_to_data_url(Path(img_path)),
        "true_description": true_desc,
        "description_for_answer": true_desc,
        "description_for_question": true_desc,
        "false_source_image_name": ""
    })

    final_rows.append({
        "tag": "false_detail",
        "image_name": img_name,
        "image_data_url": image_to_data_url(Path(img_path)),
        "true_description": true_desc,
        "description_for_answer": false_detail,
        "description_for_question": false_detail,
        "false_source_image_name": img_name
    })

    final_rows.append({
        "tag": "image",
        "image_name": img_name,
        "image_data_url": image_to_data_url(Path(img_path)),
        "true_description": true_desc,
        "description_for_answer": false_mismatch,
        "description_for_question": true_desc,
        "false_source_image_name": false_src_name
    })

    final_rows.append({
        "tag": "text",
        "image_name": img_name,
        "image_data_url": image_to_data_url(Path(img_path)),
        "true_description": true_desc,
        "description_for_answer": false_mismatch,
        "description_for_question": false_mismatch,
        "false_source_image_name": false_src_name
    })

final = pd.DataFrame(final_rows)
final.to_csv(OUT_FINAL, index=False)

In [1]:
# промпт для сценариев бинарных вопросов (image, text, concurrence)
prompt_other_tags = (
    "You need to formulate a single binary (yes/no) question based on the description of an image.\n"
    "The question must be answerable using the description.\n\n"
    "Examples:\n"
    "Description: In the picture two men in green shirts are standing in a yard.\n"
    "Question: Are there men in the picture?\n\n"
    "Description: A cat is sitting on a sofa.\n"
    "Question: Is there a cat on the sofa?\n\n"
    "Description: A person is riding a bicycle on a street.\n"
    "Question: Is there a person riding a bicycle?\n\n"
    "Formulate a question for the following description:"
)

# промпт для сценария с отличающийся деталью (false_detail)
prompt_detail = (
    "You are given two descriptions of the same image.\n"
    "They are identical except for ONE specific detail.\n\n"
    "Your task:\n"
    "1. Identify the detail in which the descriptions differ.\n"
    "2. Formulate ONE question that asks specifically about this detail.\n"
    "3. The expected answer to the question must be the VALUE of this detail "
    "(for example, a color or gender). Do NOT write any answer, only a question.\n\n"
    "Example:\n"
    "Description A: A child wearing a yellow jacket is playing in the snow.\n"
    "Description B: A child wearing a blue jacket is playing in the snow.\n\n"
    "Question: What color is the jacket the child is wearing?"
    "Formulate a question for the following descriptions:"
)



In [38]:
import pandas as pd
import asyncio
from tqdm import tqdm
from openai import AsyncOpenAI

OPENROUTER_API_KEY = ""  
MODEL = "qwen/qwen3-vl-8b-instruct"

async def generate_questions_async(
    file_in: str,
    file_out: str = None,
    concurrency: int = 16,
    save_every: int = 10,
):
    df = pd.read_csv(file_in)

    if "generated_question" not in df.columns:
        df["generated_question"] = None
    if "used_prompt" not in df.columns:
        df["used_prompt"] = None

    pending = df[df["generated_question"].isna()].copy()
    if len(pending) == 0:
        print("Вопросов нет")
        return df

    client = AsyncOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
    )

    sem = asyncio.Semaphore(concurrency)

    def build_prompt(row) -> str:
        tag = str(row["tag"]).strip()

        if tag == "false_detail":
            a = str(row["true_description"])
            b = str(row["description_for_question"])
            return (
                prompt_detail
                + f"Description A: {a}\n"
                + f"Description B: {b}\n"
                + "Question:"
            )
        else:
            d = str(row["description_for_question"])
            return (
                prompt_other_tags
                + f"Description: {d}\n"
                + "Question:"
            )

    async def process_one(index: int):
        async with sem:
            row = df.loc[index]
            prompt = build_prompt(row)
            try:
                completion = await client.chat.completions.create(
                    model=MODEL,
                    messages=[{"role": "user", "content": [{"type": "text", "text": prompt}]}],
                )
                text = completion.choices[0].message.content.strip()
                return index, prompt, text
            except Exception as e:
                print(f"Ошибка в строке {index}: {e}")
                return index, prompt, None

    tasks = [process_one(i) for i in pending.index]

    done_count = 0
    out_path = file_out if file_out else file_in

    for fut in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc="Generating"):
        index, prompt, question = await fut
        df.at[index, "generated_question"] = question
        df.at[index, "used_prompt"] = prompt
        done_count += 1

        if done_count % save_every == 0:
            df.to_csv(out_path, index=False)

    df.to_csv(out_path, index=False)
    print("Saved:", out_path)
    return df


In [39]:
df_out = await generate_questions_async(
    file_in="df_all_images.csv",
    file_out="df_questions_new.csv",
    concurrency=16,
    save_every=10
)


Generating: 100%|███████████████████████████████████████████████████████████████████| 1200/1200 [03:48<00:00,  5.25it/s]


Saved: df_questions_new.csv
